In [1]:
!pip uninstall -y delta-spark pyspark
!pip install pyspark==4.0.1 delta-spark==4.0.1

Found existing installation: pyspark 4.0.4
Uninstalling pyspark-4.0.4:
  Successfully uninstalled pyspark-4.0.4
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 434.2/434.2 MB 1.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.0/43.0 kB 1.3 MB/s eta 0:00:00
  Created wheel for pyspark: filename=pyspark-4.0.1-py2.py3-none-any.whl size=434813860 sha256=4a6026a75dfe24bf8b9905037610ee4a4adb49863891443cd189902b4e0b3925
  Stored in directory: /root/.cache/pip/wheels/00/e3/92/8594f4cee2c9fd4ad82fe85e4bf2559ab8ea84ef19b1dd3d15
Successfully built pyspark


In [2]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as sf
from pyspark.sql.types import DecimalType
from delta import configure_spark_with_delta_pip

builder = (
    SparkSession.builder
    .appName("Employee/HR Data Quality Pipeline")
    .config(
        "spark.sql.extensions",
        "io.delta.sql.DeltaSparkSessionExtension"
    )
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog"
    )
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()

#Step 1: Cewating Employee/HR Dataset

In [3]:
employee_data = [
    (" EMP1001 ", " rahul kumar ", "RAHUL.KUMAR @GMAIL.COM", "09876543210", " information technology ", "senior software engineer", "₹75,000", "2024-05-12", " active ", "MGR1001"),
    ("EMP1002", "PRIYA   SHARMA", "priya.sharma@gmail.com ", "+91-98765-43211", "Human Resources", "HR MANAGER", "85,500 INR", "12/06/2023", "ACTIVE", "MGR1002"),
    ("emp1003", " Amit Singh ", "amit.singh @ yahoo.com", "9876543212", "FINANCE", "financial analyst", "$62,500", "2023/07/18", "active", "MGR1003"),
    ("EMP1004", "neha gupta", "neha.gupta@gmail.com", " 98765 43213 ", "sales", "Sales Executive", "₹45,000.00", "18-08-2023", "Active", "MGR1004"),
    (" EMP1005", "RAJESH KUMAR", "rajesh.kumar@gmail", "+91 9876543214", "Operations", "operations manager", "1.2L", "2022-11-01", "ON LEAVE", "MGR1005"),

    ("EMP1006", "sneha patel", "SNEHA.PATEL@GMAIL.COM", "09876543215", "IT", "Data Engineer", "₹1,05,000", "01/12/2024", "active", "MGR1001"),
    ("EMP1007", "VIKAS  MEHTA", "vikas.mehta@gmail.com", "98765-43216", "Marketing", "marketing executive", "55K", "2024-01-15", "ACTIVE", "MGR1006"),
    ("EMP1008", "anita roy", "anita.roy @ outlook.com", "+919876543217", "Finance", "Senior Accountant", "₹78,500/-", "15-Jan-2023", "resigned", "MGR1003"),
    ("EMP1009", " mohit verma ", "mohit.verma@gmail.com", "9876543218", "Information Technology", "software developer", "90000", "2024.02.29", "ACTIVE", "MGR1001"),
    ("EMP1010", "KAVITA DAS", "kavita.das@gmail.com", "09876543219", "Human Resources", "Recruiter", "₹48,000", "31/02/2024", "ACTIVE", "MGR1002"),

    ("EMP1011", "arjun   nair", "arjun.nair@gmail.com", "+91-9876543220", "IT", "DevOps Engineer", "₹95,000", "2024-03-10", "active", "MGR9999"),
    ("EMP1012", "Pooja Kapoor", "pooja.kapoor@gmail.com", "9876543221", "FINANCE", "Finance Manager", "₹1,50,000", "10/04/2022", "Active", "MGR1003"),
    ("EMP1013", "Rohit Malhotra", "rohit.malhotra@gmail.com", "98765 43222", "Sales", "Sales Manager", "₹1,10,000", "2022-05-20", "active", "MGR1004"),
    ("EMP1014", "MEENA IYER", "meena.iyer@gmail.com", "+91 98765 43223", "Operations", "Operations Executive", "65000 INR", "20/06/2023", "ON LEAVE", "MGR1005"),
    ("EMP1015", "Sanjay Bose", "sanjay.bose@gmail.com", "9876543224", "Marketing", "Digital Marketing Specialist", "₹72,000", "2023-07-25", "ACTIVE", "MGR1006"),

    # Deliberately problematic records
    ("EMP10A6", "Ravi Kumar", "ravi.kumar@gmail.com", "9876543225", "IT", "Developer", "₹70,000", "2024-08-01", "ACTIVE", "MGR1001"),
    ("EMP1017", "123 Rahul", "rahul.test@gmail.com", "9876543226", "IT", "Engineer", "₹80,000", "2024-08-05", "ACTIVE", "MGR1001"),
    ("EMP1018", "Sonia@123", "sonia@gmail.com", "9876543227", "Finance", "Analyst", "₹65,000", "2024-08-10", "ACTIVE", "MGR1003"),
    ("EMP1019", "  ", "blank.name@gmail.com", "9876543228", "HR", "Recruiter", "₹50,000", "2024-08-15", "ACTIVE", "MGR1002"),
    ("EMP1020", None, "missing.name@gmail.com", "9876543229", "IT", "Tester", "₹60,000", "2024-08-20", "ACTIVE", "MGR1001"),

    ("EMP1021", "Deepak Joshi", "deepak.joshi@gmail", "9876543230", "IT", "Engineer", "₹75,000", "2024-08-21", "ACTIVE", "MGR1001"),
    ("EMP1022", "Nisha Rao", "nisha.rao @gmail.com", "9876543231", "Finance", "Analyst", "₹68,000", "2024-08-22", "ACTIVE", "MGR1003"),
    ("EMP1023", "Varun Shah", "varun.shah@gmail.com", "12345", "Sales", "Executive", "₹45,000", "2024-08-25", "ACTIVE", "MGR1004"),
    ("EMP1024", "Ayesha Khan", "ayesha.khan@gmail.com", "+91-98765-43233", "IT", "Engineer", "₹82,000", "2024-08-30", "ACTIVE", "MGR1001"),
    ("EMP1025", "Karan Singh", "karan.singh@gmail.com", "9876543234", "Unknown", "Executive", "₹52,000", "2024-09-01", "ACTIVE", "MGR1007"),

    ("EMP1026", "Ritu Sharma", "ritu.sharma@gmail.com", "9876543235", "Finance", "Accountant", "-₹55,000", "2024-09-05", "ACTIVE", "MGR1003"),
    ("EMP1027", "Manish Gupta", "manish.gupta@gmail.com", "9876543236", "IT", "Engineer", "abc", "2024-09-10", "ACTIVE", "MGR1001"),
    ("EMP1028", "Swati Sen", "swati.sen@gmail.com", "9876543237", "HR", "Recruiter", "₹45,000", "not-a-date", "ACTIVE", "MGR1002"),
    ("EMP1029", "Nitin Das", "nitin.das@gmail.com", "9876543238", "Operations", "Executive", "₹58,000", "2024-13-01", "ACTIVE", "MGR1005"),
    ("EMP1030", "Pallavi Roy", "pallavi.roy@gmail.com", "9876543239", "Marketing", "Executive", "₹61,000", "2024-10-10", "UNKNOWN", "MGR1006"),

    # More subtle problems
    ("EMP1031", "  Ajay Kumar  ", " AJAY.KUMAR@GMAIL.COM ", " +91 98765 43240 ", "Information Technology", "software engineer", "₹85,000.50", "2024-10-15", " active ", "MGR1001"),
    ("EMP1032", "SHEETAL   VERMA", "sheetal.verma@gmail.com", "09876543241", "Human  Resources", "HR Executive", "₹52,500", "2024-10-20", "Active", "MGR1002"),
    ("EMP1033", "Rakesh O'Neil", "rakesh.oneil@gmail.com", "9876543242", "Finance", "Senior Analyst", "₹88,000", "2024-10-25", "ACTIVE", "MGR1003"),
    ("EMP1034", "Mary-Anne D'Souza", "mary.annesouza@gmail.com", "9876543243", "Sales", "Sales Executive", "₹49,000", "2024-11-01", "ACTIVE", "MGR1004"),
    ("EMP1035", "  john doe  ", "JOHN.DOE@GMAIL.COM", "98765-43244", "IT", "Developer", "₹70,000", "2024-11-05", "active", "MGR1001"),

    # Duplicate employee
    ("EMP1035", "John Doe", "john.doe@gmail.com", "9876543244", "IT", "Developer", "₹70,000", "2024-11-05", "ACTIVE", "MGR1001"),

    # Null-heavy / malformed records
    (None, None, None, None, None, None, None, None, None, None),
    ("EMP1037", "Anil Kumar", None, "9876543246", "IT", "Engineer", "₹75,000", "2024-11-10", "ACTIVE", "MGR1001"),
    ("EMP1038", "Reena Das", "reena.das@gmail.com", None, "HR", "Recruiter", "₹48,000", "2024-11-12", "ACTIVE", "MGR1002"),
    ("EMP1039", "Vivek Roy", "vivek.roy@gmail.com", "9876543248", None, "Engineer", "₹80,000", "2024-11-15", "ACTIVE", "MGR1001"),
    ("EMP1040", "Tina Paul", "tina.paul@gmail.com", "9876543249", "Finance", None, "₹65,000", "2024-11-18", "ACTIVE", "MGR1003"),

    # Really tricky formats
    ("emp1041 ", "DR. Amit Kumar", "amit.kumar@gmail.com", "91-9876543250", "IT", "Tech Lead", "₹1.25 Lakh", "18-Nov-2024", "ACTIVE", "MGR1001"),
    ("EMP1042", "Lata Devi", "lata.devi@gmail.com", "0091-9876543251", "HR", "HR Executive", "75K INR", "2024/11/20", "ACTIVE", "MGR1002"),
    ("EMP1043", "Gaurav Mishra", "gaurav.mishra@gmail.com", "9876543252", "Finance", "Analyst", "1,25,000", "20-11-2024", "ACTIVE", "MGR1003"),
    ("EMP1044", "Pinky Roy", "pinky.roy@gmail.com", "98765 43253", "IT", "Data Engineer", "₹1.1L", "2024-11-22", "ACTIVE", "MGR1001"),
    ("EMP1045", "Suresh Kumar", "suresh.kumar@gmail.com", "9876543254", "Sales", "Sales Executive", "45000/-", "2024-11-25", "ACTIVE", "MGR1004"),
]

In [4]:
employee_columns = [
    "Employee_ID",
    "Employee_Name",
    "Email",
    "Phone",
    "Department",
    "Job_Title",
    "Salary",
    "Joining_Date",
    "Employee_Status",
    "Manager_ID"
]

bronze_employee_df = spark.createDataFrame(
    employee_data,
    employee_columns
)

In [5]:
bronze_employee_df.show(1)

+-----------+-------------+--------------------+-----------+--------------------+--------------------+-------+------------+---------------+----------+
|Employee_ID|Employee_Name|               Email|      Phone|          Department|           Job_Title| Salary|Joining_Date|Employee_Status|Manager_ID|
+-----------+-------------+--------------------+-----------+--------------------+--------------------+-------+------------+---------------+----------+
|   EMP1001 | rahul kumar |RAHUL.KUMAR @GMAI...|09876543210| information tech...|senior software e...|₹75,000|  2024-05-12|        active |   MGR1001|
+-----------+-------------+--------------------+-----------+--------------------+--------------------+-------+------------+---------------+----------+
only showing top 1 row


In [6]:
bronze_employee_df.count()

46

In [7]:
bronze_employee_df.printSchema()

root
 |-- Employee_ID: string (nullable = true)
 |-- Employee_Name: string (nullable = true)
 |-- Email: string (nullable = true)
 |-- Phone: string (nullable = true)
 |-- Department: string (nullable = true)
 |-- Job_Title: string (nullable = true)
 |-- Salary: string (nullable = true)
 |-- Joining_Date: string (nullable = true)
 |-- Employee_Status: string (nullable = true)
 |-- Manager_ID: string (nullable = true)



#Step 2: Bronze Layer

#A. Bronze delta path

In [8]:
project_root = "/content/delta/employee_HR"

bronze_delta_employee_path = f"{project_root}/bronze"

#B. Creating bronze employee

In [9]:
bronze_employee_df.write.format("delta").mode("overwrite").save(bronze_delta_employee_path)

#C. Reading back employee data from delta

In [10]:
bronze_delta_employee_df = spark.read.format("delta").load(bronze_delta_employee_path)

#D. Validating bronze delta table

In [11]:
bronze_delta_employee_df.printSchema()
bronze_delta_employee_df.count()
bronze_delta_employee_df.show(bronze_delta_employee_df.count())

root
 |-- Employee_ID: string (nullable = true)
 |-- Employee_Name: string (nullable = true)
 |-- Email: string (nullable = true)
 |-- Phone: string (nullable = true)
 |-- Department: string (nullable = true)
 |-- Job_Title: string (nullable = true)
 |-- Salary: string (nullable = true)
 |-- Joining_Date: string (nullable = true)
 |-- Employee_Status: string (nullable = true)
 |-- Manager_ID: string (nullable = true)

+-----------+-----------------+--------------------+-----------------+--------------------+--------------------+----------+------------+---------------+----------+
|Employee_ID|    Employee_Name|               Email|            Phone|          Department|           Job_Title|    Salary|Joining_Date|Employee_Status|Manager_ID|
+-----------+-----------------+--------------------+-----------------+--------------------+--------------------+----------+------------+---------------+----------+
|   EMP1001 |     rahul kumar |RAHUL.KUMAR @GMAI...|      09876543210| information tec

#E. Registering as a Delta table

In [12]:
spark.sql("""
    DROP TABLE IF EXISTS bronze_employee_hr
""")

spark.sql(f"""
    CREATE TABLE bronze_employee_hr
    USING DELTA
    LOCATION '{bronze_delta_employee_path}'
""")

DataFrame[]

In [13]:
spark.sql("""
    SELECT *
    FROM bronze_employee_hr
    LIMIT 10
""").show(truncate=False)

+-----------+--------------+-----------------------+---------------+------------------------+------------------------+----------+------------+---------------+----------+
|Employee_ID|Employee_Name |Email                  |Phone          |Department              |Job_Title               |Salary    |Joining_Date|Employee_Status|Manager_ID|
+-----------+--------------+-----------------------+---------------+------------------------+------------------------+----------+------------+---------------+----------+
| EMP1001   | rahul kumar  |RAHUL.KUMAR @GMAIL.COM |09876543210    | information technology |senior software engineer|₹75,000   |2024-05-12  | active        |MGR1001   |
|EMP1002    |PRIYA   SHARMA|priya.sharma@gmail.com |+91-98765-43211|Human Resources         |HR MANAGER              |85,500 INR|12/06/2023  |ACTIVE         |MGR1002   |
|emp1003    | Amit Singh   |amit.singh @ yahoo.com |9876543212     |FINANCE                 |financial analyst       |$62,500   |2023/07/18  |active  

#Step 3: Silver Layer Transformation

#A. Cleaning Employee/HR table

In [14]:
silver_employee_df = (bronze_delta_employee_df
                      .withColumn("raw_Employee_ID", sf.col("Employee_ID"))
                      .withColumn("Employee_ID",
                                  sf.upper(sf.trim("Employee_ID")))
                      .withColumn("valid_Employee_ID",
                                  sf.coalesce(
                                      sf.col("Employee_ID").rlike(r"^EMP\d+$"),
                                      sf.lit(False)))
                      .withColumn("Employee_ID",
                                  sf.when(sf.col("valid_Employee_ID"),
                                          sf.col("Employee_ID"))
                                  .otherwise(sf.lit(None)))
                      .withColumn("raw_Employee_Name", sf.col("Employee_Name"))
                      .withColumn("Employee_Name",
                                  sf.regexp_replace(
                                      sf.initcap(sf.trim("Employee_Name")),
                                      r"\s+", " "))
                      .withColumn("valid_Employee_Name",
                                  sf.coalesce(
                                      sf.col("Employee_Name")
                                      .rlike(r"^[A-Za-z]+\.?(?:[-'\s][A-Za-z]+)*$"),
                                      sf.lit(False)))
                      .withColumn("Employee_Name",
                                  sf.when(sf.col("valid_Employee_Name"),
                                          sf.col("Employee_Name"))
                                  .otherwise(sf.lit(None)))
                      .withColumn("raw_Email", sf.col("Email"))
                      .withColumn("Email",
                                  sf.regexp_replace(
                                      sf.lower(sf.trim("Email")), r"\s+", ""))
                      .withColumn("valid_Email",
                                  sf.coalesce(
                                      sf.col("Email").rlike(
                                          r"^[a-z]+(?:[-\.][a-z]+)*@[a-z]+(?:\.[a-z]+)+$"),
                                      sf.lit(False)))
                      .withColumn("Email",
                                  sf.when(sf.col("valid_Email"), sf.col("Email"))
                                  .otherwise(sf.lit(None)))
                      .withColumn("raw_Phone", sf.col("Phone"))
                      .withColumn("Phone",
                                  sf.regexp_replace(sf.regexp_replace(
                                      sf.trim("Phone"), r"\D", ""),
                                                    r"^(0091|91|0)", ""))
                      .withColumn("valid_Phone",
                                  sf.coalesce(
                                      sf.col("Phone").rlike(r"^[6-9]\d{9}$"),
                                      sf.lit(False)))
                      .withColumn("Phone",
                                  sf.when(sf.col("valid_Phone"), sf.col("Phone"))
                                  .otherwise(sf.lit(None)))
                      .withColumn("raw_Department", sf.col("Department"))
                      .withColumn("Department",
                                  sf.regexp_replace(
                                      sf.initcap(sf.trim("Department")), r"\s+", " "))
                      .withColumn("Department",
                                  sf.when(
                                      sf.col("Department").rlike(r"^(?i)IT$"),
                                      "Information Technology")
                                  .when(
                                      sf.col("Department").rlike(r"^(?i)HR$"),
                                      "Human Resources")
                                  .otherwise(sf.col("Department")))
                      .withColumn("valid_Department",
                                  sf.coalesce(
                                      sf.col("Department")
                                      .isin("Information Technology", "Human Resources",
                                            "Finance", "Sales", "Operations"),
                                      sf.lit(False)))
                      .withColumn("Department",
                                  sf.when(sf.col("valid_Department"), sf.col("Department"))
                                  .otherwise(sf.lit(None)))
                      .withColumn("raw_Job_Title", sf.col("Job_Title"))
                      .withColumn("Job_Title",
                                  sf.regexp_replace(
                                      sf.initcap(sf.trim("Job_Title")), r"\s+", " "))
                      .withColumn("valid_Job_Title",
                                  sf.coalesce(
                                      sf.col("Job_Title")
                                      .rlike(r"^[A-Za-z]+(?:[-'\.\s][A-Za-z]+)*$"),
                                      sf.lit(False)))
                      .withColumn("Job_Title",
                                  sf.when(sf.col("valid_Job_Title"), sf.col("Job_Title"))
                                  .otherwise(sf.lit(None)))
                      .withColumn("raw_Salary", sf.col("Salary"))
                      .withColumn("Salary",
                                  sf.regexp_replace(sf.regexp_replace(
                                      sf.upper(sf.trim("Salary")), r"\s+", ""),
                                                    r"[₹/,$]|INR|USD|\-$", ""))
                      .withColumn("Salary",
                                  sf.when(sf.col("Salary").rlike(r"K$"),
                                          sf.regexp_extract(sf.col("Salary"),
                                                            r"^(\d+(?:\.\d+)?)K$", 1)
                                          .try_cast("decimal(18,2)")*1000)
                                  .when(sf.col("Salary").rlike(r"(?:LAKH|L)$"),
                                          sf.regexp_extract(sf.col("Salary"),
                                                            r"^(\d+(?:\.\d+)?)", 0)
                                          .try_cast("decimal(18,2)")*100000)
                                  .otherwise(sf.col("Salary").try_cast("double")))
                      .withColumn("valid_Salary",
                                  sf.coalesce(sf.col("Salary") > 0, sf.lit(False)))
                      .withColumn("Salary",
                                  sf.when(sf.col("valid_Salary"), sf.col("Salary"))
                                  .otherwise(sf.lit(None)))
                      .withColumn("raw_Joining_Date", sf.col("Joining_Date"))
                      .withColumn("Joining_Date",
                                  sf.trim("Joining_Date"))
                      .withColumn("Joining_Date",
                                  sf.coalesce(
                                      sf.try_to_timestamp(sf.col("Joining_Date"),
                                              sf.lit("yyyy-MM-dd")).try_cast("date"),
                                      sf.try_to_timestamp(sf.col("Joining_Date"),
                                              sf.lit("yyyy/MM/dd")).try_cast("date"),
                                      sf.try_to_timestamp(sf.col("Joining_Date"),
                                              sf.lit("dd-MM-yyyy")).try_cast("date"),
                                      sf.try_to_timestamp(sf.col("Joining_Date"),
                                              sf.lit("dd/MM/yyyy")).try_cast("date"),
                                      sf.try_to_timestamp(sf.col("Joining_Date"),
                                              sf.lit("dd-MMM-yyyy")).try_cast("date"),
                                      sf.try_to_timestamp(sf.col("Joining_Date"),
                                              sf.lit("yyyy.MM.dd")).try_cast("date")))
                      .withColumn("valid_Joining_Date",
                                  sf.coalesce(sf.col("Joining_Date").isNotNull(),
                                              sf.lit(False)))
                      .withColumn("raw_Employee_Status", sf.col("Employee_Status"))
                      .withColumn("Employee_Status",
                                  sf.regexp_replace(
                                      sf.initcap(sf.trim("Employee_Status")),
                                      r"\s+", " "))
                      .withColumn("valid_Employee_Status",
                                  sf.coalesce(
                                      sf.col("Employee_Status")
                                      .isin("Active", "Inactive", "On Leave", "Terminated"),
                                      sf.lit(False)))
                      .withColumn("Employee_Status",
                                  sf.when(sf.col("valid_Employee_Status"),
                                          sf.col("Employee_Status"))
                                  .otherwise(sf.lit(None)))
                      .withColumn("raw_Manager_ID", sf.col("Manager_ID"))
                      .withColumn("Manager_ID", sf.upper(sf.trim("Manager_ID")))
                      .withColumn("valid_Manager_ID",
                                  sf.coalesce(
                                      sf.col("Manager_ID").rlike(r"^MGR\d+$"),
                                      sf.lit(False)))
                      .withColumn("Manager_ID",
                                  sf.when(sf.col("valid_Manager_ID"),
                                          sf.col("Manager_ID"))
                                  .otherwise(sf.lit(None)))
                      )

In [15]:
duplicate_employees = (silver_employee_df
                       .groupBy(sf.col("raw_Employee_ID").alias("duplicate_Employee_ID"))
                       .count()
                       .filter((sf.col("count")>1) &
                        (sf.col("duplicate_Employee_ID").isNotNull()))
                       .select("duplicate_Employee_ID")
                       )

#B. Marking Duplicate Employees

In [16]:
silver_employee_df = (silver_employee_df
                      .join(duplicate_employees,
                       (sf.col("raw_Employee_ID") ==
                        sf.col("duplicate_Employee_ID")), how="left")
                      .withColumn("is_duplicate_Employee_ID",
                                  sf.coalesce(sf.col("duplicate_Employee_ID")
                                  .isNotNull(), sf.lit(False)))
                      )

In [17]:
silver_employee_df.show(50)

+-----------+-----------------+--------------------+----------+--------------------+--------------------+--------+------------+---------------+----------+---------------+-----------------+-----------------+-------------------+--------------------+-----------+-----------------+-----------+--------------------+----------------+--------------------+---------------+----------+------------+----------------+------------------+-------------------+---------------------+--------------+----------------+---------------------+------------------------+
|Employee_ID|    Employee_Name|               Email|     Phone|          Department|           Job_Title|  Salary|Joining_Date|Employee_Status|Manager_ID|raw_Employee_ID|valid_Employee_ID|raw_Employee_Name|valid_Employee_Name|           raw_Email|valid_Email|        raw_Phone|valid_Phone|      raw_Department|valid_Department|       raw_Job_Title|valid_Job_Title|raw_Salary|valid_Salary|raw_Joining_Date|valid_Joining_Date|raw_Employee_Status|valid_Empl

#C. Identifying Invalid Records

In [18]:
silver_employee_df.printSchema()

root
 |-- Employee_ID: string (nullable = true)
 |-- Employee_Name: string (nullable = true)
 |-- Email: string (nullable = true)
 |-- Phone: string (nullable = true)
 |-- Department: string (nullable = true)
 |-- Job_Title: string (nullable = true)
 |-- Salary: double (nullable = true)
 |-- Joining_Date: date (nullable = true)
 |-- Employee_Status: string (nullable = true)
 |-- Manager_ID: string (nullable = true)
 |-- raw_Employee_ID: string (nullable = true)
 |-- valid_Employee_ID: boolean (nullable = false)
 |-- raw_Employee_Name: string (nullable = true)
 |-- valid_Employee_Name: boolean (nullable = false)
 |-- raw_Email: string (nullable = true)
 |-- valid_Email: boolean (nullable = false)
 |-- raw_Phone: string (nullable = true)
 |-- valid_Phone: boolean (nullable = false)
 |-- raw_Department: string (nullable = true)
 |-- valid_Department: boolean (nullable = false)
 |-- raw_Job_Title: string (nullable = true)
 |-- valid_Job_Title: boolean (nullable = false)
 |-- raw_Salary: st

In [19]:
silver_employee_df = (silver_employee_df
                      .withColumn("Validation_Failed",
                                  ~(sf.coalesce(sf.col("valid_Employee_ID"), sf.lit(False)) &
                                    sf.coalesce(sf.col("valid_Employee_Name"), sf.lit(False)) &
                                    sf.coalesce(sf.col("valid_Email"), sf.lit(False)) &
                                    sf.coalesce(sf.col("valid_Phone"), sf.lit(False)) &
                                    sf.coalesce(sf.col("valid_Department"), sf.lit(False)) &
                                    sf.coalesce(sf.col("valid_Job_Title"), sf.lit(False)) &
                                    sf.coalesce(sf.col("valid_Salary"), sf.lit(False)) &
                                    sf.coalesce(sf.col("valid_Joining_Date"), sf.lit(False)) &
                                    sf.coalesce(sf.col("valid_Employee_Status"), sf.lit(False)) &
                                    sf.coalesce(sf.col("valid_Manager_ID"), sf.lit(False))) &
                                  ~sf.col("is_duplicate_Employee_ID"))
                      )

In [20]:
silver_employee_df.show()

+-----------+--------------+--------------------+----------+--------------------+--------------------+--------+------------+---------------+----------+---------------+-----------------+-----------------+-------------------+--------------------+-----------+---------------+-----------+--------------------+----------------+--------------------+---------------+----------+------------+----------------+------------------+-------------------+---------------------+--------------+----------------+---------------------+------------------------+-----------------+
|Employee_ID| Employee_Name|               Email|     Phone|          Department|           Job_Title|  Salary|Joining_Date|Employee_Status|Manager_ID|raw_Employee_ID|valid_Employee_ID|raw_Employee_Name|valid_Employee_Name|           raw_Email|valid_Email|      raw_Phone|valid_Phone|      raw_Department|valid_Department|       raw_Job_Title|valid_Job_Title|raw_Salary|valid_Salary|raw_Joining_Date|valid_Joining_Date|raw_Employee_Status|va

#Step 4: Creating Quarantine table

#A. Quarantine Employee_ID

In [21]:
quarantined_employee_id_df = (silver_employee_df
                              .filter(~sf.col("valid_Employee_ID"))
                              .select(sf.col("raw_Employee_ID").alias("Employee_ID"),
                                      sf.lit("Employee_ID").alias("Column_Name"),
                                      sf.col("raw_Employee_ID").alias("Invalid_Value"),
                                      sf.lit("DQ001").alias("Rule_ID"),
                                      sf.lit("Invalid Employee_ID format")
                                      .alias("Failure_Reason"))
                              )

In [22]:
quarantined_employee_id_df.show()

+-----------+-----------+-------------+-------+--------------------+
|Employee_ID|Column_Name|Invalid_Value|Rule_ID|      Failure_Reason|
+-----------+-----------+-------------+-------+--------------------+
|    EMP10A6|Employee_ID|      EMP10A6|  DQ001|Invalid Employee_...|
|       NULL|Employee_ID|         NULL|  DQ001|Invalid Employee_...|
+-----------+-----------+-------------+-------+--------------------+



#B. Quarantine Employee_Name

In [23]:
quarantined_employee_name_df = (silver_employee_df
                              .filter(~sf.col("valid_Employee_Name"))
                              .select(sf.col("raw_Employee_ID").alias("Employee_ID"),
                                      sf.lit("Employee_Name").alias("Column_Name"),
                                      sf.col("raw_Employee_Name").alias("Invalid_Value"),
                                      sf.lit("DQ002").alias("Rule_ID"),
                                      sf.lit("Invalid Employee_Name format")
                                      .alias("Failure_Reason"))
                              )

In [24]:
quarantined_employee_name_df.show()

+-----------+-------------+-------------+-------+--------------------+
|Employee_ID|  Column_Name|Invalid_Value|Rule_ID|      Failure_Reason|
+-----------+-------------+-------------+-------+--------------------+
|    EMP1017|Employee_Name|    123 Rahul|  DQ002|Invalid Employee_...|
|    EMP1018|Employee_Name|    Sonia@123|  DQ002|Invalid Employee_...|
|    EMP1019|Employee_Name|             |  DQ002|Invalid Employee_...|
|    EMP1020|Employee_Name|         NULL|  DQ002|Invalid Employee_...|
|       NULL|Employee_Name|         NULL|  DQ002|Invalid Employee_...|
+-----------+-------------+-------------+-------+--------------------+



#C. Quarantine Email

In [25]:
quarantined_email_df = (silver_employee_df
                              .filter(~sf.col("valid_Email"))
                              .select(sf.col("raw_Employee_ID").alias("Employee_ID"),
                                      sf.lit("Email").alias("Column_Name"),
                                      sf.col("raw_Email").alias("Invalid_Value"),
                                      sf.lit("DQ003").alias("Rule_ID"),
                                      sf.lit("Invalid Email format")
                                      .alias("Failure_Reason"))
                              )

In [26]:
quarantined_email_df.show()

+-----------+-----------+------------------+-------+--------------------+
|Employee_ID|Column_Name|     Invalid_Value|Rule_ID|      Failure_Reason|
+-----------+-----------+------------------+-------+--------------------+
|    EMP1005|      Email|rajesh.kumar@gmail|  DQ003|Invalid Email format|
|    EMP1021|      Email|deepak.joshi@gmail|  DQ003|Invalid Email format|
|       NULL|      Email|              NULL|  DQ003|Invalid Email format|
|    EMP1037|      Email|              NULL|  DQ003|Invalid Email format|
+-----------+-----------+------------------+-------+--------------------+



#D. Quarantine Phone

In [27]:
quarantined_phone_df = (silver_employee_df
                              .filter(~sf.col("valid_Phone"))
                              .select(sf.col("raw_Employee_ID").alias("Employee_ID"),
                                      sf.lit("Phone").alias("Column_Name"),
                                      sf.col("raw_Phone").alias("Invalid_Value"),
                                      sf.lit("DQ004").alias("Rule_ID"),
                                      sf.lit("Invalid Phone format")
                                      .alias("Failure_Reason"))
                              )

In [29]:
quarantined_phone_df.show()

+-----------+-----------+-------------+-------+--------------------+
|Employee_ID|Column_Name|Invalid_Value|Rule_ID|      Failure_Reason|
+-----------+-----------+-------------+-------+--------------------+
|    EMP1023|      Phone|        12345|  DQ004|Invalid Phone format|
|       NULL|      Phone|         NULL|  DQ004|Invalid Phone format|
|    EMP1038|      Phone|         NULL|  DQ004|Invalid Phone format|
+-----------+-----------+-------------+-------+--------------------+



#E. Quarantine Department

In [45]:
quarantined_department_df = (silver_employee_df
                              .filter(~sf.col("valid_Department"))
                              .select(sf.col("raw_Employee_ID").alias("Employee_ID"),
                                      sf.lit("Department").alias("Column_Name"),
                                      sf.col("raw_Department").alias("Invalid_Value"),
                                      sf.lit("DQ005").alias("Rule_ID"),
                                      sf.lit("Invalid Department format")
                                      .alias("Failure_Reason"))
                              )

In [46]:
quarantined_department_df.show()

+-----------+-----------+-------------+-------+--------------------+
|Employee_ID|Column_Name|Invalid_Value|Rule_ID|      Failure_Reason|
+-----------+-----------+-------------+-------+--------------------+
|    EMP1007| Department|    Marketing|  DQ005|Invalid Departmen...|
|    EMP1015| Department|    Marketing|  DQ005|Invalid Departmen...|
|    EMP1025| Department|      Unknown|  DQ005|Invalid Departmen...|
|    EMP1030| Department|    Marketing|  DQ005|Invalid Departmen...|
|       NULL| Department|         NULL|  DQ005|Invalid Departmen...|
|    EMP1039| Department|         NULL|  DQ005|Invalid Departmen...|
+-----------+-----------+-------------+-------+--------------------+



#F. Quarantine Job_Title

In [47]:
quarantined_job_title_df = (silver_employee_df
                              .filter(~sf.col("valid_Job_Title"))
                              .select(sf.col("raw_Employee_ID").alias("Employee_ID"),
                                      sf.lit("Job_Title").alias("Column_Name"),
                                      sf.col("raw_Job_Title").alias("Invalid_Value"),
                                      sf.lit("DQ006").alias("Rule_ID"),
                                      sf.lit("Invalid Job_Title format")
                                      .alias("Failure_Reason"))
                              )

In [48]:
quarantined_job_title_df.show()

+-----------+-----------+-------------+-------+--------------------+
|Employee_ID|Column_Name|Invalid_Value|Rule_ID|      Failure_Reason|
+-----------+-----------+-------------+-------+--------------------+
|       NULL|  Job_Title|         NULL|  DQ006|Invalid Job_Title...|
|    EMP1040|  Job_Title|         NULL|  DQ006|Invalid Job_Title...|
+-----------+-----------+-------------+-------+--------------------+



#G. Quarantine Salary

In [49]:
quarantined_salary_df = (silver_employee_df
                              .filter(~sf.col("valid_Salary"))
                              .select(sf.col("raw_Employee_ID").alias("Employee_ID"),
                                      sf.lit("Salary").alias("Column_Name"),
                                      sf.col("raw_Salary").alias("Invalid_Value"),
                                      sf.lit("DQ007").alias("Rule_ID"),
                                      sf.lit("Invalid Salary format")
                                      .alias("Failure_Reason"))
                              )

In [50]:
quarantined_salary_df.show()

+-----------+-----------+-------------+-------+--------------------+
|Employee_ID|Column_Name|Invalid_Value|Rule_ID|      Failure_Reason|
+-----------+-----------+-------------+-------+--------------------+
|    EMP1026|     Salary|     -₹55,000|  DQ007|Invalid Salary fo...|
|    EMP1027|     Salary|          abc|  DQ007|Invalid Salary fo...|
|       NULL|     Salary|         NULL|  DQ007|Invalid Salary fo...|
+-----------+-----------+-------------+-------+--------------------+



#H. Quarantine Joining_Date

In [51]:
quarantined_joining_date_df = (silver_employee_df
                              .filter(~sf.col("valid_Joining_Date"))
                              .select(sf.col("raw_Employee_ID").alias("Employee_ID"),
                                      sf.lit("Joining_Date").alias("Column_Name"),
                                      sf.col("raw_Joining_Date").alias("Invalid_Value"),
                                      sf.lit("DQ008").alias("Rule_ID"),
                                      sf.lit("Invalid Joining_Date format")
                                      .alias("Failure_Reason"))
                              )

In [52]:
quarantined_joining_date_df.show()

+-----------+------------+-------------+-------+--------------------+
|Employee_ID| Column_Name|Invalid_Value|Rule_ID|      Failure_Reason|
+-----------+------------+-------------+-------+--------------------+
|    EMP1010|Joining_Date|   31/02/2024|  DQ008|Invalid Joining_D...|
|    EMP1028|Joining_Date|   not-a-date|  DQ008|Invalid Joining_D...|
|    EMP1029|Joining_Date|   2024-13-01|  DQ008|Invalid Joining_D...|
|       NULL|Joining_Date|         NULL|  DQ008|Invalid Joining_D...|
+-----------+------------+-------------+-------+--------------------+



#I. Quarantine Employee_Status

In [53]:
quarantined_employee_status_df = (silver_employee_df
                              .filter(~sf.col("valid_Employee_Status"))
                              .select(sf.col("raw_Employee_ID").alias("Employee_ID"),
                                      sf.lit("Employee_Status").alias("Column_Name"),
                                      sf.col("raw_Employee_Status").alias("Invalid_Value"),
                                      sf.lit("DQ009").alias("Rule_ID"),
                                      sf.lit("Invalid Employee_Status format")
                                      .alias("Failure_Reason"))
                              )

In [54]:
quarantined_employee_status_df.show()

+-----------+---------------+-------------+-------+--------------------+
|Employee_ID|    Column_Name|Invalid_Value|Rule_ID|      Failure_Reason|
+-----------+---------------+-------------+-------+--------------------+
|    EMP1008|Employee_Status|     resigned|  DQ009|Invalid Employee_...|
|    EMP1030|Employee_Status|      UNKNOWN|  DQ009|Invalid Employee_...|
|       NULL|Employee_Status|         NULL|  DQ009|Invalid Employee_...|
+-----------+---------------+-------------+-------+--------------------+



#J. Quarantine Manager_ID

In [55]:
quarantined_manager_id_df = (silver_employee_df
                              .filter(~sf.col("valid_Manager_ID"))
                              .select(sf.col("raw_Employee_ID").alias("Employee_ID"),
                                      sf.lit("Manager_ID").alias("Column_Name"),
                                      sf.col("raw_Manager_ID").alias("Invalid_Value"),
                                      sf.lit("DQ010").alias("Rule_ID"),
                                      sf.lit("Invalid Manager_ID format")
                                      .alias("Failure_Reason"))
                              )

In [56]:
quarantined_manager_id_df.show()

+-----------+-----------+-------------+-------+--------------------+
|Employee_ID|Column_Name|Invalid_Value|Rule_ID|      Failure_Reason|
+-----------+-----------+-------------+-------+--------------------+
|       NULL| Manager_ID|         NULL|  DQ010|Invalid Manager_I...|
+-----------+-----------+-------------+-------+--------------------+



#K. Combining the quarantined records

In [58]:
quarantined_employee_df = (quarantined_employee_id_df
                           .unionByName(quarantined_employee_name_df)
                           .unionByName(quarantined_email_df)
                           .unionByName(quarantined_phone_df)
                           .unionByName(quarantined_department_df)
                           .unionByName(quarantined_job_title_df)
                           .unionByName(quarantined_salary_df)
                           .unionByName(quarantined_joining_date_df)
                           .unionByName(quarantined_employee_status_df)
                           .unionByName(quarantined_manager_id_df)
                           )

In [60]:
quarantined_employee_df.printSchema()

root
 |-- Employee_ID: string (nullable = true)
 |-- Column_Name: string (nullable = false)
 |-- Invalid_Value: string (nullable = true)
 |-- Rule_ID: string (nullable = false)
 |-- Failure_Reason: string (nullable = false)



In [59]:
quarantined_employee_df.show()

+-----------+-------------+------------------+-------+--------------------+
|Employee_ID|  Column_Name|     Invalid_Value|Rule_ID|      Failure_Reason|
+-----------+-------------+------------------+-------+--------------------+
|    EMP10A6|  Employee_ID|           EMP10A6|  DQ001|Invalid Employee_...|
|       NULL|  Employee_ID|              NULL|  DQ001|Invalid Employee_...|
|    EMP1017|Employee_Name|         123 Rahul|  DQ002|Invalid Employee_...|
|    EMP1018|Employee_Name|         Sonia@123|  DQ002|Invalid Employee_...|
|    EMP1019|Employee_Name|                  |  DQ002|Invalid Employee_...|
|    EMP1020|Employee_Name|              NULL|  DQ002|Invalid Employee_...|
|       NULL|Employee_Name|              NULL|  DQ002|Invalid Employee_...|
|    EMP1005|        Email|rajesh.kumar@gmail|  DQ003|Invalid Email format|
|    EMP1021|        Email|deepak.joshi@gmail|  DQ003|Invalid Email format|
|       NULL|        Email|              NULL|  DQ003|Invalid Email format|
|    EMP1037

In [61]:
quarantined_employee_df.groupBy("Rule_ID").count().orderBy("Rule_ID").show()

+-------+-----+
|Rule_ID|count|
+-------+-----+
|  DQ001|    2|
|  DQ002|    5|
|  DQ003|    4|
|  DQ004|    3|
|  DQ005|    6|
|  DQ006|    2|
|  DQ007|    3|
|  DQ008|    4|
|  DQ009|    3|
|  DQ010|    1|
+-------+-----+



#L. Final silver employee table

In [64]:
silver_employee_df.show(1)

+-----------+-------------+--------------------+----------+--------------------+--------------------+-------+------------+---------------+----------+---------------+-----------------+-----------------+-------------------+--------------------+-----------+-----------+-----------+--------------------+----------------+--------------------+---------------+----------+------------+----------------+------------------+-------------------+---------------------+--------------+----------------+---------------------+------------------------+-----------------+
|Employee_ID|Employee_Name|               Email|     Phone|          Department|           Job_Title| Salary|Joining_Date|Employee_Status|Manager_ID|raw_Employee_ID|valid_Employee_ID|raw_Employee_Name|valid_Employee_Name|           raw_Email|valid_Email|  raw_Phone|valid_Phone|      raw_Department|valid_Department|       raw_Job_Title|valid_Job_Title|raw_Salary|valid_Salary|raw_Joining_Date|valid_Joining_Date|raw_Employee_Status|valid_Employee

In [69]:
silver_final_employee_df = (silver_employee_df
                            .filter(~sf.col("Validation_Failed"))
                            .select("Employee_ID", "Employee_Name", "Email",
                                    "Phone", "Department", "Job_Title",
                                    "Salary", "Joining_Date", "Employee_Status",
                                    "Manager_ID")
                            .orderBy("Employee_ID")
                            )

In [70]:
silver_final_employee_df.show(50, truncate=False)

+-----------+-----------------+------------------------+----------+----------------------+------------------------+--------+------------+---------------+----------+
|Employee_ID|Employee_Name    |Email                   |Phone     |Department            |Job_Title               |Salary  |Joining_Date|Employee_Status|Manager_ID|
+-----------+-----------------+------------------------+----------+----------------------+------------------------+--------+------------+---------------+----------+
|EMP1001    |Rahul Kumar      |rahul.kumar@gmail.com   |9876543210|Information Technology|Senior Software Engineer|75000.0 |2024-05-12  |Active         |MGR1001   |
|EMP1002    |Priya Sharma     |priya.sharma@gmail.com  |9876543211|Human Resources       |Hr Manager              |85500.0 |2023-06-12  |Active         |MGR1002   |
|EMP1003    |Amit Singh       |amit.singh@yahoo.com    |9876543212|Finance               |Financial Analyst       |62500.0 |2023-07-18  |Active         |MGR1003   |
|EMP1004  

#Step 6. Creating silver delta

#A. Silver delta path

In [62]:
print(f"project root: {project_root}")

project root: /content/delta/employee_HR


In [72]:
silver_cleaned_delta_path = f"{project_root}/silver/cleaned"
silver_final_delta_path = f"{project_root}/silver/final"
silver_quarantined_delta_path = f"{project_root}/silver/quarantined"

In [73]:
silver_employee_df.write.format("delta").mode("overwrite").save(silver_cleaned_delta_path)
silver_final_employee_df.write.format("delta").mode("overwrite").save(silver_final_delta_path)
quarantined_employee_df.write.format("delta").mode("overwrite").save(silver_quarantined_delta_path)

#B. Reading back final employee delta table

In [74]:
silver_delta_employee_df = spark.read.format("delta").load(silver_final_delta_path)

In [75]:
silver_delta_employee_df.show()

+-----------+-----------------+--------------------+----------+--------------------+--------------------+--------+------------+---------------+----------+
|Employee_ID|    Employee_Name|               Email|     Phone|          Department|           Job_Title|  Salary|Joining_Date|Employee_Status|Manager_ID|
+-----------+-----------------+--------------------+----------+--------------------+--------------------+--------+------------+---------------+----------+
|    EMP1001|      Rahul Kumar|rahul.kumar@gmail...|9876543210|Information Techn...|Senior Software E...| 75000.0|  2024-05-12|         Active|   MGR1001|
|    EMP1002|     Priya Sharma|priya.sharma@gmai...|9876543211|     Human Resources|          Hr Manager| 85500.0|  2023-06-12|         Active|   MGR1002|
|    EMP1003|       Amit Singh|amit.singh@yahoo.com|9876543212|             Finance|   Financial Analyst| 62500.0|  2023-07-18|         Active|   MGR1003|
|    EMP1004|       Neha Gupta|neha.gupta@gmail.com|9876543213|       

#C. Validating final silver data

In [76]:
silver_delta_employee_df.printSchema()
silver_delta_employee_df.show()

root
 |-- Employee_ID: string (nullable = true)
 |-- Employee_Name: string (nullable = true)
 |-- Email: string (nullable = true)
 |-- Phone: string (nullable = true)
 |-- Department: string (nullable = true)
 |-- Job_Title: string (nullable = true)
 |-- Salary: double (nullable = true)
 |-- Joining_Date: date (nullable = true)
 |-- Employee_Status: string (nullable = true)
 |-- Manager_ID: string (nullable = true)

+-----------+-----------------+--------------------+----------+--------------------+--------------------+--------+------------+---------------+----------+
|Employee_ID|    Employee_Name|               Email|     Phone|          Department|           Job_Title|  Salary|Joining_Date|Employee_Status|Manager_ID|
+-----------+-----------------+--------------------+----------+--------------------+--------------------+--------+------------+---------------+----------+
|    EMP1001|      Rahul Kumar|rahul.kumar@gmail...|9876543210|Information Techn...|Senior Software E...| 75000.0| 

In [79]:
print("Employees:", silver_delta_employee_df.count())

print("Duplicate Customer IDs:",
      silver_delta_employee_df
      .groupBy("Employee_ID")
      .count()
      .filter(sf.col("count") > 1)
      .count())

Employees: 23
Duplicate Customer IDs: 1


#Step 7: Gold Layer Transformation

In [85]:
silver_delta_employee_df.printSchema()

root
 |-- Employee_ID: string (nullable = true)
 |-- Employee_Name: string (nullable = true)
 |-- Email: string (nullable = true)
 |-- Phone: string (nullable = true)
 |-- Department: string (nullable = true)
 |-- Job_Title: string (nullable = true)
 |-- Salary: double (nullable = true)
 |-- Joining_Date: date (nullable = true)
 |-- Employee_Status: string (nullable = true)
 |-- Manager_ID: string (nullable = true)



In [107]:
gold_employee_df = (silver_delta_employee_df
                    .withColumn("Employee_Tenure",
                                sf.round(sf.datediff(
                                    sf.current_date(),
                                    sf.col("Joining_Date"))/365, 2))
                    .withColumn("Salary_Band",
                                sf.when(sf.col("Salary") < 30000, "Low")
                                .when((sf.col("Salary") >= 30000) &
                                      (sf.col("Salary") < 60000), "Lower-Middle")
                                .when((sf.col("Salary") >= 60000) &
                                      (sf.col("Salary") < 100000), "Middle")
                                .when((sf.col("Salary") >- 100000) &
                                      (sf.col("Salary") < 150000), "Upper-Middle")
                                .otherwise("High"))
                    .withColumn("Experience_Category",
                                sf.when(sf.col("Employee_Tenure") < 2, "Fresher")
                                .when((sf.col("Employee_Tenure") >= 2) &
                                      (sf.col("Employee_Tenure") < 5), "Early Career")
                                .when((sf.col("Employee_Tenure") >= 5) &
                                      (sf.col("Employee_Tenure") < 10), "Mid Career")
                                .otherwise("Experiencd"))
                    )

#Step 8: Removing duplicates

In [111]:
gold_employee_df = gold_employee_df.dropDuplicates()

In [112]:
gold_employee_df.filter(sf.col("Employee_ID") == "EMP1035").show()

+-----------+-------------+------------------+----------+--------------------+---------+-------+------------+---------------+----------+---------------+-----------+-------------------+
|Employee_ID|Employee_Name|             Email|     Phone|          Department|Job_Title| Salary|Joining_Date|Employee_Status|Manager_ID|Employee_Tenure|Salary_Band|Experience_Category|
+-----------+-------------+------------------+----------+--------------------+---------+-------+------------+---------------+----------+---------------+-----------+-------------------+
|    EMP1035|     John Doe|john.doe@gmail.com|9876543244|Information Techn...|Developer|70000.0|  2024-11-05|         Active|   MGR1001|           1.87|     Middle|            Fresher|
+-----------+-------------+------------------+----------+--------------------+---------+-------+------------+---------------+----------+---------------+-----------+-------------------+



#Step 9: Final gold table validations

#A. Final Schema nad Count verification

In [113]:
gold_employee_df.printSchema()
gold_employee_df.count()

root
 |-- Employee_ID: string (nullable = true)
 |-- Employee_Name: string (nullable = true)
 |-- Email: string (nullable = true)
 |-- Phone: string (nullable = true)
 |-- Department: string (nullable = true)
 |-- Job_Title: string (nullable = true)
 |-- Salary: double (nullable = true)
 |-- Joining_Date: date (nullable = true)
 |-- Employee_Status: string (nullable = true)
 |-- Manager_ID: string (nullable = true)
 |-- Employee_Tenure: double (nullable = true)
 |-- Salary_Band: string (nullable = false)
 |-- Experience_Category: string (nullable = false)



22

#B. Duplicate check

In [114]:
gold_employee_df.groupBy("Employee_ID").count().filter(sf.col("count")>1).show()

+-----------+-----+
|Employee_ID|count|
+-----------+-----+
+-----------+-----+



#C. Critical field's Null check

In [118]:
gold_employee_df.filter(sf.col("Employee_ID").isNotNull()).count()
gold_employee_df.filter(sf.col("Employee_Name").isNotNull()).count()
gold_employee_df.filter(sf.col("Email").isNotNull()).count()
gold_employee_df.filter(sf.col("Phone").isNotNull()).count()
gold_employee_df.filter(sf.col("Department").isNotNull()).count()
gold_employee_df.filter(sf.col("Job_Title").isNotNull()).count()
gold_employee_df.filter(sf.col("Salary").isNotNull()).count()
gold_employee_df.filter(sf.col("Joining_Date").isNotNull()).count()
gold_employee_df.filter(sf.col("Employee_Status").isNotNull()).count()
gold_employee_df.filter(sf.col("Manager_ID").isNotNull()).count()

22

#D. Derived fields

In [120]:
(gold_employee_df
 .select("Employee_ID", "Employee_Tenure", "Salary_Band", "Experience_Category")
 .show(gold_employee_df.count(), truncate=False)
 )

+-----------+---------------+------------+-------------------+
|Employee_ID|Employee_Tenure|Salary_Band |Experience_Category|
+-----------+---------------+------------+-------------------+
|EMP1009    |2.55           |Middle      |Early Career       |
|EMP1022    |2.07           |Middle      |Early Career       |
|EMP1006    |1.8            |Upper-Middle|Fresher            |
|EMP1045    |1.81           |Lower-Middle|Fresher            |
|EMP1014    |3.25           |Middle      |Early Career       |
|EMP1033    |1.9            |Middle      |Fresher            |
|EMP1043    |1.83           |Upper-Middle|Fresher            |
|EMP1035    |1.87           |Middle      |Fresher            |
|EMP1013    |4.33           |Upper-Middle|Early Career       |
|EMP1004    |3.09           |Lower-Middle|Early Career       |
|EMP1042    |1.83           |Middle      |Fresher            |
|EMP1034    |1.88           |Lower-Middle|Fresher            |
|EMP1024    |2.05           |Middle      |Early Career 

#Step 19: Writing gold data into delta

#A. Gold path

In [121]:
gold_delta_employee_path = f"{project_root}/gold"

#B. Writing into delta

In [122]:
gold_employee_df.write.format("delta").mode("overwrite").save(gold_delta_employee_path)

#C. Reading back from delta

In [123]:
gold_delta_employee_df = spark.read.format("delta").load(gold_delta_employee_path)

In [124]:
gold_delta_employee_df.show()

+-----------+-----------------+--------------------+----------+--------------------+--------------------+--------+------------+---------------+----------+---------------+------------+-------------------+
|Employee_ID|    Employee_Name|               Email|     Phone|          Department|           Job_Title|  Salary|Joining_Date|Employee_Status|Manager_ID|Employee_Tenure| Salary_Band|Experience_Category|
+-----------+-----------------+--------------------+----------+--------------------+--------------------+--------+------------+---------------+----------+---------------+------------+-------------------+
|    EMP1009|      Mohit Verma|mohit.verma@gmail...|9876543218|Information Techn...|  Software Developer| 90000.0|  2024-02-29|         Active|   MGR1001|           2.55|      Middle|       Early Career|
|    EMP1022|        Nisha Rao| nisha.rao@gmail.com|9876543231|             Finance|             Analyst| 68000.0|  2024-08-22|         Active|   MGR1003|           2.07|      Middle| 